# 00 - Python, NumPy, Matplotlib, Pandas, and PySpark Bootcamp

This is your **starting notebook** for the project.

Goal: go from fundamentals to production-ready understanding for this transport analytics project.

You will learn:
1. Python foundations
2. NumPy for vectorized numeric computing
3. Matplotlib for plots and visual debugging
4. Pandas for tabular/time-series analytics
5. PySpark foundations for big-data scale
6. Practice + active recall in each section

## How to use this notebook

- Run cells top to bottom.
- For practice cells, try first before reading the provided solution/check.
- Keep this notebook as your repeated training notebook.

## Project Explanation (Read This First)

This repository has **two data folders on purpose**.

### `data/` and `datasets/` (important)

`data/`:

- Main working storage for project data.
- Contains large/raw source files (like MTA) and processed outputs (`data/processed/...`).
- Think: pipeline working area.

`datasets/`:

- Curated external CSV datasets used as clean input sources.
- Usually smaller and more structured for analysis.
- Think: curated input package.

So:

- `data/` = raw + processed working data
- `datasets/` = curated dataset inputs

### Why both folders exist

- We keep **large/raw operational files** in `data/` because they change and can be very heavy.
- We keep **clean/selected research inputs** in `datasets/` so analysis notebooks can start quickly.
- We write pipeline outputs to `data/processed/` to avoid mixing raw and transformed data.

### Practical project flow

1. Raw/curated files are read from `data/` + `datasets/`.
2. Cleaning and schema harmonization happen in pipeline code.
3. Canonical fact tables are created in `data/processed/`.
4. SQL tables/views are applied in PostgreSQL (via pgAdmin web).
5. Analysis/forecasting notebooks use processed tables and SQL outputs.

### Where to look when you are new

- Start learning here: `00_python_numpy_matplotlib_pandas_pyspark_bootcamp.ipynb`
- Project analytics notebook: `01_data_analytics_pipeline.ipynb`
- SQL and DS foundations: `03_data_science_big_data_sql_foundations.ipynb`
- SQL practice: `04_sql_active_learning_practice.ipynb`
- PostgreSQL setup (pgAdmin web): `docs/POSTGRES_SETUP.md`

In [ ]:
# Standard imports used throughout the notebook.
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Global notebook display settings.
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
plt.style.use("seaborn-v0_8-whitegrid")

# Random generator with fixed seed for reproducibility.
rng = np.random.default_rng(42)

print("Environment ready.")

---
## Part A - Python Fundamentals

In [ ]:
# Variables and basic types.
city = "Paris"
riders_today = 15423
avg_delay_minutes = 3.7
is_weekend = False

# Collections.
line_codes = ["M1", "M4", "RER A"]
station_to_region = {"Chatelet": "Ile-de-France", "Times Sq": "NYC"}

print(type(city), type(riders_today), type(avg_delay_minutes), type(is_weekend))
print(line_codes)
print(station_to_region)

In [ ]:
# Control flow and looping.
weekly_counts = [12000, 11800, 12500, 13000, 12700, 9800, 9200]

weekday_total = 0
weekend_total = 0

for i, value in enumerate(weekly_counts):
    if i < 5:
        weekday_total += value
    else:
        weekend_total += value

print("weekday_total:", weekday_total)
print("weekend_total:", weekend_total)
print("weekend ratio:", round(weekend_total / (weekday_total + weekend_total), 3))

In [ ]:
# List comprehension and dictionary comprehension.
values = [12, 0, 4, 9, 0, 18, 7]

# Keep only non-zero values.
non_zero = [v for v in values if v != 0]

# Build index map for quick lookup.
idx_map = {idx: v for idx, v in enumerate(values)}

print("non_zero:", non_zero)
print("idx_map sample:", {k: idx_map[k] for k in [0, 3, 6]})

In [ ]:
# Functions with type hints and docstrings.
def pct_change(new_value: float, old_value: float) -> float:
    """Return percentage change from old_value to new_value."""
    if old_value == 0:
        return float("nan")
    return (new_value - old_value) / old_value

print("pct_change(13000, 12000):", round(pct_change(13000, 12000), 4))

In [ ]:
# Dataclass example for clear transport records.
@dataclass
class DemandPoint:
    date: str
    station: str
    riders: int

    def is_high_demand(self, threshold: int = 12000) -> bool:
        # Encapsulated logic improves readability and reuse.
        return self.riders >= threshold

p = DemandPoint(date="2024-01-10", station="Chatelet", riders=14200)
print(p)
print("high_demand:", p.is_high_demand())

### Python Practice (with checker)

Task:
- Write a function `moving_average(values, window)` returning a list of rolling means.
- Example: `[2, 4, 6, 8]` with `window=2` -> `[3.0, 5.0, 7.0]`

In [ ]:
# TODO: Replace this with your own implementation first.
def moving_average(values: list[float], window: int) -> list[float]:
    # Reference solution (you can overwrite).
    if window <= 0 or window > len(values):
        return []
    out = []
    for i in range(window - 1, len(values)):
        chunk = values[i - window + 1 : i + 1]
        out.append(sum(chunk) / window)
    return out

# Simple checker.
assert moving_average([2, 4, 6, 8], 2) == [3.0, 5.0, 7.0]
assert moving_average([1, 1, 1], 3) == [1.0]
print("Python practice check passed.")

### Active Recall - Python

Answer mentally first, then reveal.

In [ ]:
recall_cards_python = [
    ("What is a list comprehension?", "Compact syntax to create lists from iterables with optional filtering."),
    ("Why use a dataclass?", "To define lightweight classes with auto-generated init/repr and clearer structure."),
    ("Difference between list and tuple?", "List is mutable, tuple is immutable."),
    ("What does `if __name__ == '__main__'` do?", "Runs code only when file is executed directly, not imported."),
]

def draw_cards(cards, reveal=False):
    rows = []
    for q, a in cards:
        rows.append({"question": q, "answer": a if reveal else "(hidden)"})
    return pd.DataFrame(rows)

draw_cards(recall_cards_python, reveal=False)

---
## Part B - NumPy Foundations

In [ ]:
# Create arrays with explicit dtype and shape.
a = np.array([1, 2, 3, 4, 5], dtype=np.float64)
b = np.arange(12).reshape(3, 4)

print("a:", a)
print("a dtype:", a.dtype)
print("b:\n", b)
print("b shape:", b.shape)


In [ ]:
# Indexing, slicing, and boolean masks.
arr = np.array([10, 15, 8, 30, 22, 7, 14])

print("arr[0]:", arr[0])
print("arr[2:5]:", arr[2:5])

# Keep values above threshold.
mask = arr > 12
print("mask:", mask)
print("filtered:", arr[mask])

In [ ]:
# Vectorization and broadcasting.
base = np.array([100, 200, 300, 400])
growth_rate = 1.05

# Vectorized operation is faster than Python loops for large arrays.
next_period = base * growth_rate
print("next_period:", next_period)

# Broadcasting: add one offset per row.
matrix = np.arange(12).reshape(3, 4)
offsets = np.array([100, 200, 300]).reshape(3, 1)
print("matrix + offsets:\n", matrix + offsets)


In [ ]:
# Statistics and linear algebra basics.
x = rng.normal(loc=1000, scale=120, size=500)

print("mean:", round(x.mean(), 2))
print("std:", round(x.std(), 2))
print("p10/p50/p90:", np.percentile(x, [10, 50, 90]))

# Dot product example for weighted score.
weights = np.array([0.4, 0.35, 0.25])
features = np.array([0.8, 0.6, 0.9])
print("weighted score:", float(np.dot(weights, features)))

In [ ]:
# NumPy practice + checker.
# Task: normalize array with z-score: (x - mean) / std
def zscore(v: np.ndarray) -> np.ndarray:
    # Reference solution.
    return (v - v.mean()) / v.std()

test = np.array([1.0, 2.0, 3.0, 4.0])
out = zscore(test)
assert np.isclose(out.mean(), 0.0)
assert np.isclose(out.std(), 1.0)
print("NumPy practice check passed.")

---
## Part C - Matplotlib Foundations

In [ ]:
# Build synthetic daily demand series for plotting examples.
dates = pd.date_range("2024-01-01", periods=60, freq="D")
demand_fr = 12000 + 1200 * np.sin(np.linspace(0, 3 * np.pi, 60)) + rng.normal(0, 300, 60)
demand_us = 17000 + 1400 * np.sin(np.linspace(0, 3 * np.pi, 60) + 0.5) + rng.normal(0, 350, 60)

demo_df = pd.DataFrame({"date": dates, "fr": demand_fr, "us": demand_us})
demo_df.head()

In [ ]:
# Line chart for trend comparison.
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(demo_df["date"], demo_df["fr"], label="Ile-de-France")
ax.plot(demo_df["date"], demo_df["us"], label="NYC")
ax.set_title("Daily demand trend")
ax.set_xlabel("Date")
ax.set_ylabel("Riders")
ax.legend()
plt.show()

In [ ]:
# Histogram + scatter + bar in one figure.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(demo_df["fr"], bins=15, color="#1f77b4")
axes[0].set_title("FR demand distribution")

axes[1].scatter(demo_df["fr"], demo_df["us"], alpha=0.7)
axes[1].set_title("FR vs US demand")
axes[1].set_xlabel("FR")
axes[1].set_ylabel("US")

weekly = demo_df.copy()
weekly["week"] = weekly["date"].dt.isocalendar().week.astype(int)
weekly = weekly.groupby("week", as_index=False)[["fr", "us"]].mean()
axes[2].bar(weekly["week"].astype(str), weekly["fr"], label="FR")
axes[2].set_title("Weekly avg FR demand")
axes[2].tick_params(axis="x", rotation=60)

plt.tight_layout()
plt.show()

In [ ]:
# Matplotlib practice prompt:
# Build your own plot of rolling 7-day mean and compare with raw values.

temp = demo_df.copy()
temp["fr_roll7"] = temp["fr"].rolling(7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(temp["date"], temp["fr"], alpha=0.4, label="raw")
ax.plot(temp["date"], temp["fr_roll7"], linewidth=2, label="roll7")
ax.set_title("Practice solution: rolling mean")
ax.legend()
plt.show()

---
## Part D - Pandas Foundations

In [ ]:
# Series and DataFrame basics.
s = pd.Series([100, 120, 90], name="riders")

df = pd.DataFrame(
    {
        "station": ["A", "B", "C", "A", "B", "C"],
        "date": pd.date_range("2024-01-01", periods=6, freq="D"),
        "riders": [100, 130, 90, 115, 128, 97],
        "tickets": [80, 100, 70, 92, 101, 76],
    }
)

print(s)
df

In [ ]:
# Filtering, selecting, and assignment.
high = df[df["riders"] > 110].copy()
high["ratio_ticket_to_riders"] = high["tickets"] / high["riders"]

high

In [ ]:
# Groupby and aggregation patterns.
by_station = (
    df.groupby("station", as_index=False)
      .agg(
          mean_riders=("riders", "mean"),
          max_riders=("riders", "max"),
          sum_tickets=("tickets", "sum"),
      )
)

by_station

In [ ]:
# Merge / join patterns in pandas.
station_meta = pd.DataFrame(
    {
        "station": ["A", "B", "C"],
        "line": ["L1", "L1", "L2"],
        "region": ["FR", "FR", "US"],
    }
)

joined = df.merge(station_meta, on="station", how="left")
joined.head()

In [ ]:
# Time-series operations: resample and rolling.
ts = joined.set_index("date").sort_index()
weekly = ts.resample("W")["riders"].sum().to_frame("weekly_riders")
weekly["roll2"] = weekly["weekly_riders"].rolling(2, min_periods=1).mean()

weekly

In [ ]:
# Load real project CSV sample to connect training with project data.
root = Path("..").resolve()
sample_path = root / "datasets" / "Travel_titles_validations_in_Paris_and_suburbs.csv"

# Use nrows to keep notebook responsive.
real_sample = pd.read_csv(sample_path, nrows=20000)

# Basic cleanup for the sample.
real_sample["DATE"] = pd.to_datetime(real_sample["DATE"], dayfirst=True, errors="coerce")
real_sample["NB_VALID_NUM"] = (
    real_sample["NB_VALID"]
    .astype(str)
    .str.replace("Less than 5", "2", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

real_daily = real_sample.groupby("DATE", as_index=False)["NB_VALID_NUM"].sum()
real_daily.head()

In [ ]:
# Visualize real sample behavior.
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(real_daily["DATE"], real_daily["NB_VALID_NUM"])
ax.set_title("Real project sample: daily validations")
ax.set_xlabel("Date")
ax.set_ylabel("Validations")
plt.show()

In [ ]:
# Pandas practice checker.
# Task: produce a DataFrame with station + total riders sorted descending.
def station_totals(input_df: pd.DataFrame) -> pd.DataFrame:
    # Reference solution.
    out = (
        input_df.groupby("station", as_index=False)["riders"]
        .sum()
        .rename(columns={"riders": "total_riders"})
        .sort_values("total_riders", ascending=False)
        .reset_index(drop=True)
    )
    return out

st = station_totals(df)
assert list(st.columns) == ["station", "total_riders"]
assert st["total_riders"].iloc[0] >= st["total_riders"].iloc[-1]
print("Pandas practice check passed.")
st

---
## Part E - PySpark Foundations (Big Data)

PySpark matters when data is too big for a single-machine pandas workflow.

In [ ]:
# Detect whether PySpark is available.
import importlib.util

spark_available = importlib.util.find_spec("pyspark") is not None
print("pyspark available:", spark_available)

In [ ]:
# PySpark section with safe fallback if pyspark is not installed.
if spark_available:
    # Import Spark only when available.
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F

    # Create local Spark session for demonstration.
    spark = SparkSession.builder.master("local[*]").appName("Bootcamp00").getOrCreate()

    # Convert pandas training DataFrame to Spark DataFrame.
    sdf = spark.createDataFrame(joined[["station", "date", "riders", "tickets", "line", "region"]])

    # Typical Spark transformations: select/filter/group/order.
    spark_result = (
        sdf.groupBy("region", "line")
        .agg(
            F.avg("riders").alias("avg_riders"),
            F.sum("riders").alias("sum_riders"),
        )
        .orderBy(F.desc("sum_riders"))
    )

    spark_result.show()
else:
    # Fallback explanation + pandas equivalent so notebook remains runnable.
    print("PySpark not installed in this environment.")
    print("Install later with: pip install pyspark")
    print("Equivalent pandas pattern shown below:")

    fallback = (
        joined.groupby(["region", "line"], as_index=False)
        .agg(avg_riders=("riders", "mean"), sum_riders=("riders", "sum"))
        .sort_values("sum_riders", ascending=False)
    )
    display(fallback)

In [ ]:
# Spark mental model: lazy transformations and actions.
spark_model = pd.DataFrame(
    {
        "concept": ["Transformation", "Action", "Lazy execution", "Partitioning"],
        "example": ["select, filter, withColumn", "count, show, collect", "Plan executes on action", "Parallel chunks of data"],
        "why_it_matters": [
            "Build pipeline steps",
            "Trigger computation",
            "Optimize distributed execution",
            "Scale workload over cluster",
        ],
    }
)

spark_model

### PySpark Practice Prompt

Write Spark code (or pandas fallback) to compute:
- total riders per region per day
- 7-day rolling average per region

You can use the patterns from previous sections.

---
## Part F - Mini Project Workflow (from this repository)

In [ ]:
# Build a mini end-to-end pipeline using real sampled project data.
root = Path("..").resolve()
idfm_path = root / "datasets" / "idfm_validations_surface.csv"

# The file uses ';' separator.
idfm = pd.read_csv(idfm_path, sep=";", nrows=50000)

# Basic cleaning for project-ready columns.
idfm["JOUR"] = pd.to_datetime(idfm["JOUR"], errors="coerce")
idfm["NB_VALD"] = pd.to_numeric(idfm["NB_VALD"], errors="coerce")

# Daily region-like aggregation at line level.
idfm_daily = (
    idfm.groupby(["JOUR", "LIBELLE_LIGNE"], as_index=False)["NB_VALD"]
    .sum()
    .rename(columns={"JOUR": "date", "LIBELLE_LIGNE": "location_name", "NB_VALD": "value"})
)

# Add training-style features.
idfm_daily = idfm_daily.sort_values(["location_name", "date"])
idfm_daily["dow"] = idfm_daily["date"].dt.dayofweek
idfm_daily["lag_1"] = idfm_daily.groupby("location_name")["value"].shift(1)
idfm_daily["roll7"] = idfm_daily.groupby("location_name")["value"].transform(lambda s: s.rolling(7, min_periods=2).mean())

idfm_daily.head()

In [ ]:
# Plot one location to understand trend + rolling behavior.
top_line = (
    idfm_daily.groupby("location_name", as_index=False)["value"].sum()
    .sort_values("value", ascending=False)
    .iloc[0]["location_name"]
)

plot_df = idfm_daily[idfm_daily["location_name"] == top_line].copy()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(plot_df["date"], plot_df["value"], alpha=0.45, label="raw")
ax.plot(plot_df["date"], plot_df["roll7"], linewidth=2, label="roll7")
ax.set_title(f"Mini project view - {top_line}")
ax.legend()
plt.show()

---
## Part G - Active Learning and Revision System

In [ ]:
# Unified recall deck across all topics.
recall_deck = pd.DataFrame(
    [
        ("Python", "What problem do type hints solve?", "They improve readability, tooling, and early error detection."),
        ("NumPy", "What is broadcasting?", "Applying operations on arrays with compatible shapes without manual loops."),
        ("Matplotlib", "When to use scatter vs line?", "Scatter for relationship points; line for ordered trend/time series."),
        ("Pandas", "What does groupby-agg do?", "Splits data by keys then computes summary metrics."),
        ("PySpark", "What is lazy execution?", "Spark defers computation until an action is called."),
        ("Project", "Why create lag features?", "To give models access to past behavior for forecasting."),
    ],
    columns=["topic", "question", "answer"],
)

def practice_cards(deck: pd.DataFrame, n: int = 4, reveal: bool = False, seed: int = 0) -> pd.DataFrame:
    sample = deck.sample(n=min(n, len(deck)), random_state=seed).reset_index(drop=True)
    if not reveal:
        sample = sample.copy()
        sample["answer"] = "(hidden)"
    return sample

practice_cards(recall_deck, n=5, reveal=False, seed=2)

In [ ]:
# Mini multiple-choice quiz with auto-scoring.
quiz = [
    {
        "q": "Which library is best for single-machine tabular analytics?",
        "options": ["A) NumPy", "B) Pandas", "C) Matplotlib", "D) PySpark SQL only"],
        "answer": "B",
    },
    {
        "q": "Which operation triggers Spark execution?",
        "options": ["A) filter", "B) withColumn", "C) show", "D) select"],
        "answer": "C",
    },
    {
        "q": "What does a 7-day rolling mean help with?",
        "options": ["A) Random shuffling", "B) Smoothing noise", "C) Encoding strings", "D) SQL parsing"],
        "answer": "B",
    },
]

user_answers = ["B", "C", "B"]  # Replace with your answers.

score = 0
for i, item in enumerate(quiz):
    print(f"Q{i+1}: {item['q']}")
    print(" ".join(item["options"]))
    print("Your answer:", user_answers[i], "| Correct:", item["answer"])
    if user_answers[i].upper() == item["answer"]:
        score += 1
    print("---")

print(f"Score: {score}/{len(quiz)}")

## Suggested path after this notebook

1. `01_data_analytics_pipeline.ipynb`
2. `02_papers_summary.ipynb`
3. `03_data_science_big_data_sql_foundations.ipynb`
4. `04_sql_active_learning_practice.ipynb`

You can revisit this `00` notebook anytime for revision.